# Gapminder 1: Data Manipulation
**ENGR 1451 / 2451 · Exploratory Data Analytics**

**Focus:** load → filter and select → create columns → summarize → save.

Keep `data/gapminder.csv` in a `data` folder beside this notebook. Run from top to bottom;
before saving your work, restart the kernel and run all cells. This notebook runs independently
of Parts 2 and 3. The exercise is left for you to complete.

## 1. Load and inspect the data
Each row is a **country–year observation**, not a person. This historical excerpt contains
**1,704 rows: 142 countries at 12 time points, from 1952 through 2007**.

| Column | Meaning |
|:--|:--|
| `country` | Historical country label |
| `continent` | Continent group |
| `year` | Observation year |
| `lifeExp` | Life expectancy at birth, in years |
| `pop` | Population, in people |
| `gdpPercap` | GDP per capita, in inflation-adjusted US dollars |

In [ ]:
from pathlib import Path
import pandas as pd

gapminder = pd.read_csv(Path("data/gapminder.csv"))
print(f"Rows: {gapminder.shape[0]:,}; columns: {gapminder.shape[1]}")
gapminder.head()

In [ ]:
gapminder.info()
gapminder[["lifeExp", "pop", "gdpPercap"]].describe().round(2)

## 2. Filter, select, and sort
Use `.loc[row_condition, column_names]` to select rows and columns together.
Combine conditions with `&` (**and**) or `|` (**or**), with parentheses around each condition.
Use `==` to compare values.

In [ ]:
gapminder.loc[
    (gapminder["country"] == "Rwanda") & (gapminder["year"] > 1979),
    ["country", "year", "lifeExp"],
]

In [ ]:
# A one-year snapshot: one row per country.
gap_2007 = gapminder.loc[gapminder["year"] == 2007].copy()

# Method chaining: filter, then sort, then show the first five rows.
(
    gap_2007.loc[:, ["country", "continent", "lifeExp", "gdpPercap"]]
    .sort_values("lifeExp", ascending=False)
    .head(5)
)

## 3. Create columns
`assign()` returns a new DataFrame. Column arithmetic is vectorized.
Here population is converted to millions and total GDP to billions of the dataset's dollars.

In [ ]:
gap_2007 = gap_2007.assign(
    pop_millions=gap_2007["pop"] / 1_000_000,
    gdp_billions=gap_2007["pop"] * gap_2007["gdpPercap"] / 1_000_000_000,
)

gap_2007[["country", "pop_millions", "gdpPercap", "gdp_billions"]].head()

## 4. Group and summarize
Each named aggregation has the form `new_name=("column", "operation")`.
`agg()` returns one row per continent. These 2007 summaries give each country equal weight;
they are **not population-weighted**. `std` uses the sample standard deviation, with denominator $n-1$.

In [ ]:
continent_summary = (
    gap_2007.groupby("continent", as_index=False)
    .agg(
        n_countries=("country", "nunique"),
        mean_life_exp=("lifeExp", "mean"),
        median_life_exp=("lifeExp", "median"),
        s_life_exp=("lifeExp", "std"),
    )
)
continent_summary.round(2)

### Exercise 1 — Build a useful comparison table
For **2007**, select countries in the **Americas** with populations greater than **10 million**.
Show `country`, `lifeExp`, `pop_millions`, and `gdpPercap`, sorted from highest to lowest life expectancy.
Does the country with the highest life expectancy also have the highest GDP per capita?

In [ ]:
# Your code here.

**Your interpretation:**

_Write one or two sentences here._

## 5. Save the summary table
`index=False` omits the row-index column. Rerunning this cell replaces the generated CSV,
not the input data.

In [ ]:
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)
table_path = output_dir / "gapminder_continent_summary_2007.csv"
continent_summary.to_csv(table_path, index=False)
print(f"Saved: {table_path}")

---
## Command reference
| Task | Python pattern |
|:--|:--|
| Read a CSV | `pd.read_csv(path)` |
| Inspect | `df.head()`, `df.info()`, `df.describe()` |
| Filter and select | `df.loc[condition, ["country", "lifeExp"]]` |
| Sort | `df.sort_values("lifeExp", ascending=False)` |
| Create a column | `df.assign(pop_millions=df["pop"] / 1e6)` |
| Group and summarize | `df.groupby("continent").agg(mean_life_exp=("lifeExp", "mean"))` |
| Save a CSV | `df.to_csv(path, index=False)` |
